In [ ]:
#@title Vidomoka voice engine v6: press ▶ (Run), then allow Google Drive
# VIDEO STUDIO VOICE ENGINE - press Run (▶) and allow Google Drive. Nothing to change here.
# Turn on the graphics card first: Runtime > Change runtime type > T4 GPU.
"""Vidomoka voice engine (UPGRADE v49) - makes custom voices with Qwen3-TTS.

The SAME file runs in two places:
  * Google Colab (fast, free GPU): Vidomoka turns it into a one-cell notebook. You press Run once, allow
    Google Drive, and it works through the voice jobs Vidomoka puts in your Drive folder "Video Studio voices".
  * This computer (no Colab, slower): Vidomoka starts it by itself in the background.
Voices are cloned the way Qwen recommends for the closest match (accent, rhythm, tone): the voice's sample AND the
exact words said in it ("ICL" mode). The voice is prepared once and reused for every line of the script, so a whole
video keeps one consistent voice. Nothing is kept anywhere except your own Drive / your own computer.
"""
import glob
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import traceback
import wave

WORKER_VERSION = 6                    # v100.7: Colab by link too (Colab 1, 2, 3: no Google Drive needed)
FOLDER = "Video Studio voices"
BASE_GPU = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
BASE_CPU = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
DESIGN = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
HEARTBEAT = 15


# ---------------------------------------------------------------- small helpers
def _rj(p, default=None):
    try:
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default


def _wj(p, data):
    tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1)
    os.replace(tmp, p)


def _md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def _write_wav(path, audio, sr):
    import numpy as np
    a = np.asarray(audio, dtype=np.float32).reshape(-1)
    pcm = (np.clip(a, -1, 1) * 32767).astype(np.int16)
    tmp = path + ".part"
    with wave.open(tmp, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(int(sr))
        w.writeframes(pcm.tobytes())
    os.replace(tmp, path)


class Screen:
    """One big status line: in Colab it updates in place; on this computer it prints."""

    def __init__(self, colab):
        self.colab, self.h = colab, None

    def show(self, icon, text, small=""):
        if self.colab:
            from IPython.display import HTML, display
            html = (f"<div style='font:600 22px/1.35 system-ui,Segoe UI,Arial;padding:14px 18px;border-radius:12px;"
                    f"background:#111827;color:#F9FAFB'>{icon} {text}"
                    + (f"<div style='font:400 14px/1.4 system-ui;opacity:.75;margin-top:6px'>{small}</div>" if small else "")
                    + "</div>")
            if self.h is None:
                self.h = display(HTML(html), display_id=True)
            else:
                self.h.update(HTML(html))
        else:
            print(f"{icon} {text}" + (f"  ({small})" if small else ""), flush=True)


# ---------------------------------------------------------------- setting up
def install(fast, cache, screen, colab):
    try:
        import qwen_tts  # noqa
        return
    except ImportError:
        pass
    wheels = os.path.join(cache, "wheels")
    if colab and not shutil.which("sox"):
        subprocess.run("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y sox libsox-dev >/dev/null 2>&1", shell=True)
    if fast and glob.glob(os.path.join(wheels, "qwen_tts*")):
        screen.show("⏳", "Installing the voice engine from your Drive (fast start)...")
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", wheels, "qwen-tts"],
                           capture_output=True, text=True)
        if r.returncode == 0:
            return
    screen.show("⏳", "Installing the voice engine (about 1-2 minutes)...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qwen-tts", "soundfile"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Couldn't install the voice engine: " + (r.stderr or r.stdout)[-300:])
    if fast:                                           # keep the installers in Drive for next time (in the background)
        def keep():
            os.makedirs(wheels, exist_ok=True)
            subprocess.run([sys.executable, "-m", "pip", "download", "-q", "-d", wheels, "qwen-tts", "soundfile"],
                           capture_output=True, text=True)
        threading.Thread(target=keep, daemon=True).start()


def model_path(name, fast, cache, screen):
    """A model folder: from your Drive when fast start has kept it there, else downloaded (and kept, if asked)."""
    local = os.path.join(cache, "models", name.replace("/", "__"))
    if fast and os.path.exists(os.path.join(local, "config.json")) and os.path.exists(os.path.join(local, ".complete")):
        return local
    from huggingface_hub import snapshot_download
    p = snapshot_download(name)
    if fast:
        def keep():
            try:
                tmp = local + ".part"
                shutil.rmtree(tmp, ignore_errors=True)
                shutil.copytree(p, tmp)
                shutil.rmtree(local, ignore_errors=True)
                os.replace(tmp, local)
                open(os.path.join(local, ".complete"), "w").close()
            except Exception:
                pass
        threading.Thread(target=keep, daemon=True).start()
    return p


# ---------------------------------------------------------------- the worker
class Worker:
    def __init__(self, root, device, colab, screen, base_model, gpu="", devices=None, dtype=None, where=None):
        self.root, self.device, self.colab, self.screen = root, device, colab, screen
        self.gpu = gpu
        self.base_name = base_model
        self.base = self.design = None
        self.devices = devices or (["cuda:0"] if device == "cuda" else ["cpu"])
        self.dtype = dtype                                   # "bf16" | "fp16" | None (bf16 on a graphics card)
        self.where = where or ("colab" if colab else "computer")
        self.bases = {}                                      # v97.9: one cloning model per graphics card
        self.prompts = {}
        self.mlock = threading.Lock()
        self.state = {"state": "starting", "msg": "", "job": None, "done": 0, "total": 0}
        self.settings = _rj(os.path.join(root, "settings.json"), {}) or {}
        self.cache = os.path.join(root, "cache")
        for sub in ("jobs", "results", "voices"):
            os.makedirs(os.path.join(root, sub), exist_ok=True)
        threading.Thread(target=self._beat, daemon=True).start()

    def _beat(self):
        while True:
            try:
                _wj(os.path.join(self.root, "worker.json"), dict(self.state, t=time.time(), gpu=self.gpu, device=self.device,
                                                                 version=WORKER_VERSION, where=self.where, cards=len(self.devices)))
            except Exception:
                pass
            time.sleep(HEARTBEAT)

    def set(self, state, msg, **kw):
        self.state.update(state=state, msg=msg, **kw)

    def _load(self, name, dev=None):
        import torch
        from qwen_tts import Qwen3TTSModel
        fast = bool(self.settings.get("fast_start"))
        path = model_path(name, fast, self.cache, self.screen)
        dev = dev or self.devices[0]
        if dev.startswith("cuda"):
            dt = torch.float16 if self.dtype == "fp16" else torch.bfloat16
            return Qwen3TTSModel.from_pretrained(path, device_map=dev, dtype=dt, attn_implementation="eager")
        return Qwen3TTSModel.from_pretrained(path, device_map="cpu", dtype=torch.float32, attn_implementation="eager")

    def base_model(self, dev=None):
        dev = dev or self.devices[0]
        with self.mlock:
            if dev not in self.bases:
                self.set("loading", "Loading the cloning model...")
                self.screen.show("⏳", "Loading the voice model" + (f" on graphics card {self.devices.index(dev) + 1}" if len(self.devices) > 1 else "") + "...")
                self.bases[dev] = self._load(self.base_name, dev)
            self.base = self.bases[self.devices[0]] if self.devices[0] in self.bases else self.bases[dev]
            return self.bases[dev]

    def design_model(self):
        with self.mlock:
            if self.design is None:
                self.set("loading", "Loading the voice design model...")
                self.screen.show("⏳", "Loading the voice design model (the first design takes a minute)...")
                self.design = self._load(DESIGN, self.devices[-1])
            return self.design

    def prompt(self, vid, dev=None):
        """The voice, prepared once from its sample and the words said in it."""
        vd = os.path.join(self.root, "voices", vid)
        meta = _rj(os.path.join(vd, "voice.json"), {}) or {}
        sample = os.path.join(vd, "sample.wav")
        if not os.path.exists(sample):
            raise RuntimeError(f"The voice '{meta.get('name') or vid}' hasn't reached this Drive yet. Wait a minute.")
        dev = dev or self.devices[0]
        key = (vid, _md5(sample), meta.get("transcript"), dev)
        if key not in self.prompts:
            m = self.base_model(dev)
            text = (meta.get("transcript") or "").strip()
            if text:
                self.prompts[key] = m.create_voice_clone_prompt(ref_audio=sample, ref_text=text)
            else:                                     # no transcript: only the voice's colour (less close)
                self.prompts[key] = m.create_voice_clone_prompt(ref_audio=sample, ref_text=None, x_vector_only_mode=True)
        return self.prompts[key]

    def jobs(self):
        out = []
        for p in glob.glob(os.path.join(self.root, "jobs", "*.json")):
            j = _rj(p)
            if not j or not j.get("id"):
                continue
            rd = os.path.join(self.root, "results", j["id"])
            if os.path.exists(os.path.join(rd, "done")) or os.path.exists(os.path.join(rd, "error.txt")):
                continue
            out.append((j.get("created", 0), p, j))
        return [x[1:] for x in sorted(out)]

    def run_job(self, path, job):
        """Makes the parts not made yet (only the ones in job["only"], when it says). With two graphics cards, each
        takes the next part waiting, so a voice-over takes about half the time."""
        import numpy as np
        try:
            import torch
        except ImportError:                                # (the tests run without it)
            torch = None
        rd = os.path.join(self.root, "results", job["id"])
        os.makedirs(rd, exist_ok=True)
        chunks = job.get("chunks") or []
        n = len(chunks)
        only = set(int(i) for i in job["only"]) if job.get("only") is not None else None
        lang = job.get("language") or "Auto"
        kw = {"temperature": float(job.get("temperature", 0.8)), "max_new_tokens": int(job.get("max_new_tokens", 2048))}
        label = job.get("label") or "voice"
        todo = [i for i in range(n) if (only is None or i in only) and not os.path.exists(os.path.join(rd, f"{i:04d}.wav"))]
        t0, made, lock, err = time.time(), [0], threading.Lock(), []
        q = list(todo)
        design = job.get("kind") == "design"
        devs = self.devices[:1] if design else self.devices
        if not design:
            for d in devs:                                 # each card gets the voice ready first (once)
                self.prompt(job["voice"], d)

        def one(dev):
            while True:
                with lock:
                    if err or not q:
                        return
                    i = q.pop(0)
                    left = len(q) + 1
                out = os.path.join(rd, f"{i:04d}.wav")
                if not os.path.exists(path):              # Vidomoka cancelled this job
                    with lock:
                        q.clear()
                    return
                with lock:
                    el = time.time() - t0
                    eta = f", about {int((el / made[0]) * left / max(1, len(devs)) / 60) + 1} min left" if made[0] else ""
                    k = n - len(q)
                self.set("working", f"Making voice: {label}, part {k} of {n}", job=job["id"], done=k - 1, total=n)
                self.screen.show("🎙️", f"Making voice: {label}, part {k} of {n}{eta}",
                                 "Keep this tab open. You can keep working in Vidomoka." if self.where != "kaggle" else
                                 (f"On {len(devs)} graphics cards at once." if len(devs) > 1 else ""))
                try:
                    if torch is not None:
                        torch.manual_seed(int(job.get("seed", 1234)) + i)
                    if design:
                        wavs, sr = self.design_model().generate_voice_design(text=chunks[i]["text"], instruct=job.get("instruct") or "",
                                                                             language=lang, **kw)
                    else:
                        wavs, sr = self.base_model(dev).generate_voice_clone(text=chunks[i]["text"], language=lang,
                                                                             voice_clone_prompt=self.prompt(job["voice"], dev), **kw)
                    a = wavs[0]
                    if hasattr(a, "cpu"):
                        a = a.cpu().numpy()
                    a = np.squeeze(np.asarray(a, dtype=np.float32))
                    if not np.isfinite(a).all():
                        raise RuntimeError("The voice came out broken (the graphics card's number format). Restart the engine.")
                    _write_wav(out, a, sr)
                    with lock:
                        made[0] += 1
                except Exception as e:
                    with lock:
                        err.append(e)
                    return

        if len(devs) > 1 and len(todo) > 1:
            th = [threading.Thread(target=one, args=(d,), daemon=True) for d in devs]
            for x in th:
                x.start()
            for x in th:
                x.join()
        else:
            one(devs[0])
        if err:
            raise err[0]
        if not os.path.exists(path):
            self.set("ready", "Cancelled.")
            return
        if all(os.path.exists(os.path.join(rd, f"{i:04d}.wav")) for i in range(n) if only is None or i in only):
            open(os.path.join(rd, "done"), "w").close()
        took = time.time() - t0
        self.set("ready", f"Finished: {label}.", job=None, done=n, total=n, last_took=round(took, 1), last_parts=len(todo))

    def stop_asked(self):
        """Vidomoka's Stop Colab button leaves stop.json in the folder."""
        p = os.path.join(self.root, "stop.json")
        s = _rj(p)
        if s and float(s.get("t", 0)) > self.started - 5:
            try:
                os.remove(p)
            except Exception:
                pass
            return True
        return False

    def loop(self, idle_exit=None):
        self.started = time.time()
        self.set("ready", "Ready.")
        self.screen.show("✅", f"Ready (voice engine v{WORKER_VERSION}). Go back to Vidomoka.", "Keep this tab open while Vidomoka makes voices.")
        self.idle = time.time()
        while True:
            self.settings = _rj(os.path.join(self.root, "settings.json"), {}) or self.settings
            todo = self.jobs()
            for path, job in todo:
                try:
                    self.run_job(path, job)
                except Exception as e:
                    rd = os.path.join(self.root, "results", job["id"])
                    os.makedirs(rd, exist_ok=True)
                    msg = str(e).strip().splitlines()[-1] if str(e).strip() else type(e).__name__
                    if "out of memory" in str(e).lower():
                        msg = "The graphics card ran out of memory. Runtime > Restart session, then Run again."
                    with open(os.path.join(rd, "error.txt"), "w", encoding="utf-8") as f:
                        f.write(msg[:500] + "\n\n" + traceback.format_exc()[-2000:])
                    self.set("ready", "Error: " + msg[:200])
                self.screen.show("✅", f"Ready (voice engine v{WORKER_VERSION}). Go back to Vidomoka.", "Keep this tab open while Vidomoka makes voices.")
                self.idle = time.time()
            if self.stop_asked():
                self.set("stopped", "Stopped from Vidomoka.")
                _wj(os.path.join(self.root, "worker.json"), dict(self.state, t=time.time(), version=WORKER_VERSION))
                self.screen.show("⏹️", "Stopped from Vidomoka. You can close this tab.",
                                 "The graphics card is given back to Google. Press ▶ Run again to start it next time.")
                self.stopped = True
                return
            if idle_exit:
                self.stop_in = int(idle_exit - (time.time() - self.idle))
            if idle_exit and time.time() - self.idle > idle_exit:
                self.set("stopped", "Stopped (nothing to do).")
                _wj(os.path.join(self.root, "worker.json"), dict(self.state, t=0))
                return
            time.sleep(3)


# ---------------------------------------------------------------- the two ways to start it
def colab_main():
    screen = Screen(True)
    screen.show("⏳", f"Starting the Vidomoka voice engine v{WORKER_VERSION}...")
    try:
        import torch
        if not torch.cuda.is_available():
            screen.show("⚠️", "This Colab has no graphics card turned on.",
                        "Click Runtime → Change runtime type → T4 GPU → Save, then press Run again.")
            return
        gpu = torch.cuda.get_device_name(0)
    except Exception as e:
        screen.show("⚠️", "Colab isn't ready: " + str(e)[:120])
        return
    screen.show("🔑", "Allow Google Drive in the window that opens...", "Vidomoka and this notebook talk through your Drive.")
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:                            # the Google Drive permission didn't go through
        msg = str(e)
        screen.show("⚠️", "Google Drive wasn't allowed, so Vidomoka can't talk to this notebook.",
                    "Fix: Runtime → Restart session, press ▶ Run again, choose the SAME Google account as this Colab "
                    "(top right), tick <b>Select all</b> on the permissions page, then Continue. No window? Allow pop-ups "
                    "for this site. Still failing? Open this notebook in an Incognito window with only that one account."
                    + (f"<br><span style='opacity:.6'>({msg[:120]})</span>" if msg else ""))
        return
    root = os.path.join("/content/drive/MyDrive", FOLDER)
    os.makedirs(root, exist_ok=True)
    settings = _rj(os.path.join(root, "settings.json"), {}) or {}
    fast = bool(settings.get("fast_start"))
    try:
        install(fast, os.path.join(root, "cache"), screen, True)
        w = Worker(root, "cuda", True, screen, BASE_GPU, gpu)
        w.base_model()
    except Exception as e:
        screen.show("⚠️", "Couldn't start: " + str(e)[:200], "Press Run again. If it happens again, Runtime → Disconnect and delete runtime, then Run.")
        raise
    w.loop()
    if getattr(w, "stopped", False):                  # Stop Colab: end this session so Google frees the graphics card
        try:
            time.sleep(20)                            # time for the "stopped" note to reach your Drive
            from google.colab import runtime
            runtime.unassign()
        except Exception:
            pass


# ---------------------------------------------------------------- v97.9: the web door (Kaggle has no Google Drive)
def serve(worker, code, port=7862, host="127.0.0.1"):
    """Vidomoka talks to the engine on Kaggle through this (behind a Cloudflare quick tunnel). Every call carries the
    code. The jobs go in the same folders as on Drive, so the same worker makes them."""
    import base64
    import re as _re
    from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
    root = worker.root
    safe = lambda s: bool(_re.fullmatch(r"[A-Za-z0-9_-]{1,64}", str(s or "")))

    class H(BaseHTTPRequestHandler):
        def log_message(self, *a):
            pass

        def _j(self, c, o):
            b = json.dumps(o).encode()
            self.send_response(c)
            self.send_header("content-type", "application/json")
            self.send_header("content-length", str(len(b)))
            self.end_headers()
            self.wfile.write(b)

        def _ok(self):
            if self.headers.get("x-code") != code:
                self._j(401, {"error": "wrong code"})
                return False
            return True

        def _body(self):
            n = int(self.headers.get("content-length") or 0)
            return json.loads(self.rfile.read(n) or b"{}") if n else {}

        def do_GET(self):
            if not self._ok():
                return
            p = self.path.split("?")[0].strip("/").split("/")
            if p == ["health"] or p == [""]:
                w = _rj(os.path.join(root, "worker.json"), {}) or {}
                return self._j(200, {"ok": True, "engine": "vidomoka-voice", "version": WORKER_VERSION, "state": worker.state.get("state"),
                                     "msg": worker.state.get("msg"), "gpus": worker.gpu, "cards": len(worker.devices),
                                     "dtype": worker.dtype or "", "idle_s": int(time.time() - getattr(worker, "idle", time.time())),
                                     "stop_in_s": getattr(worker, "stop_in", None), "job": w.get("job"), "done": w.get("done"), "total": w.get("total")})
            if len(p) == 2 and p[0] == "voice" and safe(p[1]):
                vd = os.path.join(root, "voices", p[1])
                s = os.path.join(vd, "sample.wav")
                return self._j(200, {"md5": _md5(s) if os.path.exists(s) else "", "meta": _rj(os.path.join(vd, "voice.json"), {}) or {}})
            if len(p) == 2 and p[0] == "job" and safe(p[1]):
                rd = os.path.join(root, "results", p[1])
                done = sorted(int(f[:4]) for f in os.listdir(rd) if _re.fullmatch(r"\d{4}\.wav", f)) if os.path.isdir(rd) else []
                er = ""
                if os.path.exists(os.path.join(rd, "error.txt")):
                    er = open(os.path.join(rd, "error.txt"), encoding="utf-8", errors="replace").read().split("\n\n")[0]
                return self._j(200, {"have": os.path.exists(os.path.join(root, "jobs", p[1] + ".json")), "done": done, "error": er,
                                     "finished": os.path.exists(os.path.join(rd, "done"))})
            if len(p) == 3 and p[0] == "part" and safe(p[1]) and p[2].isdigit():
                f = os.path.join(root, "results", p[1], f"{int(p[2]):04d}.wav")
                if not os.path.exists(f):
                    return self._j(404, {"error": "not made yet"})
                b = open(f, "rb").read()
                self.send_response(200)
                self.send_header("content-type", "audio/wav")
                self.send_header("content-length", str(len(b)))
                self.end_headers()
                self.wfile.write(b)
                return
            self._j(404, {"error": "unknown"})

        def do_POST(self):
            if not self._ok():
                return
            p = self.path.split("?")[0].strip("/").split("/")
            try:
                b = self._body()
            except Exception:
                return self._j(400, {"error": "bad request"})
            if len(p) == 2 and p[0] == "voice" and safe(p[1]):
                vd = os.path.join(root, "voices", p[1])
                os.makedirs(vd, exist_ok=True)
                with open(os.path.join(vd, "sample.wav.part"), "wb") as f:
                    f.write(base64.b64decode(b.get("sample") or ""))
                os.replace(os.path.join(vd, "sample.wav.part"), os.path.join(vd, "sample.wav"))
                _wj(os.path.join(vd, "voice.json"), {k: b.get(k) for k in ("name", "transcript", "language")})
                return self._j(200, {"ok": True})
            if p == ["job"] and safe(b.get("id")):
                rd = os.path.join(root, "results", b["id"])
                if os.path.exists(os.path.join(rd, "error.txt")):
                    os.remove(os.path.join(rd, "error.txt"))
                if os.path.exists(os.path.join(rd, "done")):
                    os.remove(os.path.join(rd, "done"))
                _wj(os.path.join(root, "jobs", b["id"] + ".json"), dict(b, created=b.get("created") or time.time()))
                return self._j(200, {"ok": True})
            if p == ["stop"]:
                _wj(os.path.join(root, "stop.json"), {"t": time.time()})
                return self._j(200, {"ok": True})
            self._j(404, {"error": "unknown"})

        def do_DELETE(self):
            if not self._ok():
                return
            p = self.path.split("?")[0].strip("/").split("/")
            if len(p) == 2 and p[0] == "job" and safe(p[1]):
                try:
                    os.remove(os.path.join(root, "jobs", p[1] + ".json"))
                except OSError:
                    pass
                shutil.rmtree(os.path.join(root, "results", p[1]), ignore_errors=True)
                return self._j(200, {"ok": True})
            self._j(404, {"error": "unknown"})

    srv = ThreadingHTTPServer((host, port), H)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    return srv


KAGGLE_IDLE = 2 * 3600                       # nothing to do for 2 hours: it stops, so your Kaggle hours aren't used up


def _tunnel(port, log):
    """A Cloudflare quick tunnel to the web door; returns its https link."""
    import re as _re
    exe = "/kaggle/tmp/cloudflared"
    os.makedirs("/kaggle/tmp", exist_ok=True)
    if not os.path.exists(exe):
        subprocess.run(f"wget -q -O {exe} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {exe}", shell=True)
    tun = subprocess.Popen([exe, "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True)
    for line in tun.stdout:
        m = _re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()   # keep reading its output
            return m.group(0)
    raise RuntimeError("The tunnel didn't start: " + log)


def kaggle_main(pair=None, precision="auto", kind="voice"):
    """v97.9: the voice engine on Kaggle (Save & Run All, or Run all). Two T4 cards: one cloning model on each.
    precision: "auto" (half precision on T4 cards, checked by a self-test), "fp16" or "bf16" (slower, always clean)."""
    return link_main(pair, precision, "kaggle", kind)


def link_main(pair=None, precision="auto", where="kaggle", kind="voice"):
    """v100.7: the voice engine behind its own link, on Kaggle or on Colab (no Google Drive needed): it tells your
    Vidomoka its link by itself (kind: "voice", or a numbered one like "voice-c2" for Colab 2), or you paste it."""
    import secrets
    import socket
    screen = Screen(False)
    say = lambda *a: print(*a, flush=True)
    colab = where == "colab"
    say(f"Vidomoka voice engine v{WORKER_VERSION} on {'Colab' if colab else 'Kaggle'}")
    gpus = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
    if not gpus:
        say("STOPPED: no graphics card. " + ("Runtime > Change runtime type > T4 GPU > Save, then run it again." if colab else
                                              "Settings (right panel) > Accelerator > GPU T4 x2, then run it again."))
        return
    for host in ("pypi.org", "huggingface.co", "github.com"):
        try:
            socket.gethostbyname(host)
        except Exception:
            say("STOPPED: no internet. Settings (right panel) > Internet: on (Kaggle asks you to verify your phone once), then run it again.")
            return
    names = gpus.splitlines()
    say("Graphics cards:", " + ".join(names))
    root = "/content/vidomoka_voice" if colab else "/kaggle/tmp/vidomoka_voice"   # the big disk (not saved as the notebook's output)
    os.environ.setdefault("HF_HOME", "/content/hf" if colab else "/kaggle/tmp/hf")
    out_dir = "/content" if colab else "/kaggle/working"
    for sub in ("jobs", "results", "voices", "cache"):
        os.makedirs(os.path.join(root, sub), exist_ok=True)
    install(False, os.path.join(root, "cache"), screen, True)
    import torch
    devs = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cuda:0"]
    # T4 cards have no fast bfloat16: half precision (fp16) is much quicker there. A short self-test proves it speaks
    # clean; if not, bfloat16 (slower, always clean).
    if precision in ("fp16", "bf16"):
        dtype = precision
    else:                                                    # T4 (and P100) have no real bfloat16
        dtype = "fp16" if torch.cuda.get_device_capability(0)[0] < 8 else "bf16"
    say(f"Precision: {dtype}" + (" (if the voice ever sounds wrong, set PRECISION = \"bf16\" at the top and run again)" if dtype == "fp16" else ""))
    w = None
    for dt in ([dtype, "bf16"] if dtype == "fp16" and precision == "auto" else [dtype]):
        w = Worker(root, "cuda", False, screen, BASE_GPU, " + ".join(names), devices=devs, dtype=dt, where=where)
        try:
            t0 = time.time()
            w.base_model(devs[0])
            say(f"Model loaded on {devs[0]} ({dt}) in {time.time() - t0:.0f} s")
            ok_ = _selftest(w, devs[0], say)
            if ok_:
                break
            say("fp16 made a broken sound: using bfloat16 instead.")
            w.bases.clear()
            torch.cuda.empty_cache()
        except Exception as e:
            say("Loading failed:", str(e)[:300])
            if dt == "bf16":
                raise
    for d in devs[1:]:
        w.base_model(d)
    code = secrets.token_hex(8)
    serve(w, code, 7862)
    link = _tunnel(7862, "")
    w.link, w.code = link, code
    import urllib.request
    t1 = time.time()                                       # a new link needs a moment before it answers
    while time.time() - t1 < 300:
        try:
            urllib.request.urlopen(urllib.request.Request(link + "/health", headers={"x-code": code}), timeout=10).read()
            break
        except Exception:
            time.sleep(10)
    say("")
    say("=" * 60)
    say("READY")
    say("Link: " + link)
    say("Code: " + code)
    say("=" * 60)
    open(os.path.join(out_dir, "LINK.txt"), "w").write(f"{link}\n{code}\n")
    try:
        from IPython.display import HTML, display
        display(HTML(f"<div style='font:600 20px/1.5 system-ui;padding:16px;border-radius:12px;background:#0f5132;color:#fff'>READY: Vidomoka voice engine<br>"
                     f"Link: <code style='color:#fff'>{link}</code><br>Code: <code style='color:#fff'>{code}</code></div>"))
    except Exception:
        pass
    if pair:
        try:
            _announce(pair, link, code, say, kind)
        except Exception as e:
            say("Couldn't tell Vidomoka by itself (" + str(e)[:120] + "): paste the link and code in Vidomoka.")
        threading.Thread(target=_reannounce, args=(pair, link, code, kind), daemon=True).start()
    w.idle = time.time()
    w.loop(idle_exit=KAGGLE_IDLE)
    say("Stopped: " + str(w.state.get("msg") or ""))
    if colab:                                              # end the Colab session so Google frees the graphics card
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception:
            pass


def _reannounce(pair, link, code, kind="voice"):
    """The link is left again every 6 hours (the server keeps it 3 days; a new copy of your app finds it too).
    v100.7: was every 10 minutes: that cost your server about 144 writes a day for each running engine."""
    while True:
        time.sleep(6 * 3600)
        try:
            _announce(pair, link, code, lambda *a: None, kind)
        except Exception:
            pass


def _selftest(w, dev, say):
    """One short line in a made-up voice: it must come out as real sound (fp16 on some cards gives silence or noise)."""
    import numpy as np
    vd = os.path.join(w.root, "voices", "_selftest")
    os.makedirs(vd, exist_ok=True)
    sr = 24000
    t_ = np.linspace(0, 2, 2 * sr, endpoint=False)
    _write_wav(os.path.join(vd, "sample.wav"), 0.2 * np.sin(2 * np.pi * 180 * t_) * (0.5 + 0.5 * np.sin(2 * np.pi * 3 * t_)), sr)
    _wj(os.path.join(vd, "voice.json"), {"name": "test", "transcript": "", "language": "English"})
    t0 = time.time()
    try:
        wavs, sr2 = w.base_model(dev).generate_voice_clone(text="This is a short test of the voice engine.", language="English",
                                                         voice_clone_prompt=w.prompt("_selftest", dev), max_new_tokens=400)
        a = wavs[0]
        if hasattr(a, "cpu"):
            a = a.cpu().numpy()
        a = np.squeeze(np.asarray(a, dtype=np.float32))
        good = bool(a.size > sr2 * 0.5 and np.isfinite(a).all() and float(np.abs(a).max()) > 0.01)
        try:
            _write_wav(("/content" if os.path.isdir("/content") and not os.path.isdir("/kaggle") else "/kaggle/working") + "/selftest.wav", a, sr2)      # to listen to
        except Exception:
            pass
        say(f"Self-test: {'OK' if good else 'broken sound'} ({a.size / sr2:.1f} s of voice in {time.time() - t0:.0f} s)")
        return good
    except Exception as e:
        say("Self-test failed:", str(e)[:300])
        return False


def _seal(key_hex, obj):
    """v97.9.1: scrambled with your app's own key (the server in between never sees the link or its code)."""
    import base64
    import hmac
    import secrets as _s
    k, nonce, data = bytes.fromhex(key_hex), _s.token_bytes(16), json.dumps(obj).encode()
    stream, c = b"", 0
    while len(stream) < len(data):
        stream += hashlib.sha256(k + nonce + c.to_bytes(4, "big")).digest()
        c += 1
    ct = bytes(a ^ b for a, b in zip(data, stream))
    return base64.urlsafe_b64encode(nonce + ct + hmac.new(k, nonce + ct, hashlib.sha256).digest()[:16]).decode()


def _unseal(key_hex, box):
    import base64
    import hmac
    raw = base64.urlsafe_b64decode(str(box).encode())
    k, nonce, ct, mac = bytes.fromhex(key_hex), raw[:16], raw[16:-16], raw[-16:]
    if not hmac.compare_digest(mac, hmac.new(k, nonce + ct, hashlib.sha256).digest()[:16]):
        raise ValueError("not ours")
    stream, c = b"", 0
    while len(stream) < len(ct):
        stream += hashlib.sha256(k + nonce + c.to_bytes(4, "big")).digest()
        c += 1
    return json.loads(bytes(a ^ b for a, b in zip(ct, stream)).decode())


def _announce(pair, link, code, say, kind="voice"):
    """v97.9.1: tells your Vidomoka its new link (through the meeting point on your server), so it connects by itself."""
    import urllib.request
    box = _seal(pair["key"], {"link": link, "code": code, "kind": kind, "t": time.time()})
    req = urllib.request.Request(pair["server"].rstrip("/") + "/pair/put", data=json.dumps({"id": pair["id"], "kind": kind, "box": box}).encode(),
                                 headers={"content-type": "application/json"}, method="POST")
    for i in range(5):
        try:
            with urllib.request.urlopen(req, timeout=20) as r:
                if json.loads(r.read() or b"{}").get("ok"):
                    say("Vidomoka was told the link: it connects by itself (nothing to paste).")
                    return True
        except Exception as e:
            err = e
            time.sleep(5 * (i + 1))
    say(f"Couldn't tell Vidomoka by itself ({str(err)[:100]}): paste the link and code in Vidomoka.")
    return False


def colab_link_cell(pair=None, kind="voice-c1", name="Colab 1"):
    """v100.7: the Colab cell for the link way (no Google Drive): paste it in a new Colab notebook and press Run."""
    with open(os.path.abspath(__file__), encoding="utf-8") as f:
        src = f.read()
    src = src.split("\nif __name__ == \"__main__\"")[0]
    head = (f"#@title Vidomoka voice engine v{WORKER_VERSION} ({name}): press Run. Nothing to change here.\n"
            "# Turn on the graphics card first: Runtime > Change runtime type > T4 GPU.\n"
            "PRECISION = \"auto\"\n")
    return head + src + f"\n\nlink_main({json.dumps(pair) if pair else 'None'}, PRECISION, \"colab\", {json.dumps(kind)})\n"


def colab_link_notebook_json(pair=None, kind="voice-c1", name="Colab 1"):
    return json.dumps({"nbformat": 4, "nbformat_minor": 0,
                       "metadata": {"accelerator": "GPU", "colab": {"provenance": [], "gpuType": "T4", "name": f"Vidomoka Voice ({name})"},
                                    "kernelspec": {"name": "python3", "display_name": "Python 3"}},
                       "cells": [{"cell_type": "markdown", "metadata": {}, "source": [
                                     f"# Vidomoka voice engine: {name}\n\n", "1. Runtime > Change runtime type > **T4 GPU** > Save.\n",
                                     "2. Runtime > **Run all**.\n", "3. In about 5 minutes it says **READY**"
                                     + (" and connects to your Vidomoka by itself" if pair else ": paste the link and code in Vidomoka (Engines)") + ".\n\n",
                                     "It stops by itself after 2 hours with nothing to do.\n"]},
                                 {"cell_type": "code", "metadata": {"cellView": "form"}, "execution_count": None, "outputs": [],
                                  "source": colab_link_cell(pair, kind, name).splitlines(True)}]}, indent=1)


def kaggle_notebook_json(pair=None, kind="voice"):
    """The Kaggle notebook: this file, then start it. `pair`: your private pairing key (it connects by itself)."""
    with open(os.path.abspath(__file__), encoding="utf-8") as f:
        src = f.read()
    src = src.split("\nif __name__ == \"__main__\"")[0]
    intro = ("# Vidomoka · Voice engine on Kaggle\n\n"
             "Your voices (My voices) made on Kaggle's free graphics cards: the backup when Colab is off, or first if you choose.\n\n"
             "**Once:** a Kaggle account with a verified phone number (that unlocks the graphics cards and the internet).\n\n"
             "**Each time:**\n"
             "1. Right panel > **Session options**: *Accelerator* = **GPU T4 x2**, *Internet* = **On**.\n"
             "2. Press **Save Version** > **Save & Run All (Commit)**. It runs in the background (it doesn't take the "
             "place of your clips notebook, and you can close the tab). Or press **Run all** if nothing else runs.\n"
             + ("3. In about 5 minutes it's READY and **connects to your Vidomoka by itself** (this copy of the notebook is yours: "
                "it carries your private pairing key; don't share it).\n\n" if pair else
                "3. In about 5 minutes the log says **READY** with a link and a code: Vidomoka's Voice tower > Voice engine "
                "> Kaggle: paste them and press Save.\n\n") +
             "It stops by itself after 2 hours with nothing to do, so your weekly Kaggle hours aren't used up.\n")
    code = (f"# Vidomoka voice engine v{WORKER_VERSION} for Kaggle: nothing to change here.\n"
            "PRECISION = \"auto\"   # \"auto\" = fast half precision on T4 cards; \"bf16\" if the voice ever sounds wrong (slower)\n" + src +
            f"\n\nkaggle_main({json.dumps(pair) if pair else 'None'}, PRECISION, {json.dumps(kind)})\n")
    cell = lambda kind, s: dict({"cell_type": kind, "metadata": {}, "source": s.splitlines(True)},
                                **({"execution_count": None, "outputs": []} if kind == "code" else {}))
    return json.dumps({"nbformat": 4, "nbformat_minor": 5,
                       "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
                                    "kaggle": {"accelerator": "nvidiaTeslaT4", "isInternetEnabled": True, "isGpuEnabled": True,
                                               "language": "python", "sourceType": "notebook"}},
                       "cells": [cell("markdown", intro), cell("code", code)]}, indent=1)


def local_main(argv):
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--model", default=BASE_CPU)
    ap.add_argument("--idle-exit", type=float, default=600)
    a = ap.parse_args(argv)
    screen = Screen(False)
    dev = "cpu"
    try:
        import torch
        if torch.cuda.is_available():
            dev = "cuda"
    except Exception:
        pass
    w = Worker(a.root, dev, False, screen, a.model if dev == "cpu" else BASE_GPU,
               "this computer's processor" if dev == "cpu" else "this computer's graphics card")
    w.loop(idle_exit=a.idle_exit)


def notebook_cell():
    """The whole Colab cell: this file, then start it."""
    with open(os.path.abspath(__file__), encoding="utf-8") as f:
        src = f.read()
    src = src.split("\nif __name__ == \"__main__\"")[0]
    head = (f"#@title Vidomoka voice engine v{WORKER_VERSION}: press ▶ (Run), then allow Google Drive\n"
            "# VIDEO STUDIO VOICE ENGINE - press Run (▶) and allow Google Drive. Nothing to change here.\n"
            "# Turn on the graphics card first: Runtime > Change runtime type > T4 GPU.\n")
    return head + src + "\n\ncolab_main()\n"


def notebook_json():
    return json.dumps({"nbformat": 4, "nbformat_minor": 0,
                       "metadata": {"accelerator": "GPU", "colab": {"provenance": [], "gpuType": "T4", "name": "Video Studio Voice"},
                                    "kernelspec": {"name": "python3", "display_name": "Python 3"}},
                       "cells": [{"cell_type": "code", "metadata": {"cellView": "form"}, "execution_count": None, "outputs": [],
                                  "source": notebook_cell().splitlines(True)}]}, indent=1)



colab_main()
